# Task 1: Third-order letter aproximation model

In [19]:
### IMPORTS
import os
from collections import defaultdict
import random
import json

In [20]:
def remove_intro_and_outro(text):
    # Find the start marker for the actual book content
    start_marker = "*** START OF"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        # Find the next line break after the start marker and cut text after it
        text = text[start_idx + len(start_marker):].strip()

    # Find the end marker for the end of the book content
    end_marker = "*** END OF"
    end_idx = text.find(end_marker)
    if end_idx != -1:
        # Cut text before the end marker
        text = text[:end_idx].strip()

    return text

#### remove_intro_and_outro() 
the above method iterates through the text file provided, and removes all text before the section of the text with __"*** START OF"__ which corelates to the section of the text that covers the intro by Project Gutenberg placed in every written work.
In the same vain the method also checks for the __"*** END OF"__ section of text to remove all the endnotes and credits after the text, this leaves only the raw written text as it would be found in any book, without added watermarks and headers from Project Gutenberg 

In [21]:
def basic_cleaning(text):    
    cleaned_text = []
    for char in text:
        if char.isalpha() or char == ' ' or char == '.':
            cleaned_text.append(char)
    return ''.join(cleaned_text)

### basic_cleaning()
the above method iterates through the text file, this time inserting all text into an array of characters, we iterate through the array checking if any characters are empty spaces, periods, or uppercase, in the instance a character does not confrom to these conditions, the character is replaced with a null char.

In [22]:
def advanced_preprocess_text(file_path):
    # Step 1: Read the file
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Step 2: Remove intro and outro
    text = remove_intro_and_outro(text)
    
    # Step 3: Convert to uppercase
    text = text.upper()

    # Step 4: Remove unwanted characters (keep letters, spaces, and full stops)
    text = basic_cleaning(text)
    
    return text

#### advanced_preprocess_text()
finally this function calls all our previous methods to take in, iterate through, and clean through all the text, before outputting a file 

In [23]:
def trigram_iteration(text, trigram_counts):
    # Iterate through the text to count trigrams
    for i in range(len(text) - 2):
        trigram = text[i:i+3]
        trigram_counts[trigram] += 1

    return trigram_counts


#### trigram_iteration()
this function is called and iterates through the text file splitting the text at every three characters to create a trigram

# ===================================================================================
# Task 2: Third-order letter aproximation generation

In [24]:
def get_possible_trigrams(trigram_model, last_two_chars):
    possible_trigrams = {}
    for trigram, count in trigram_model.items():
        if trigram.startswith(last_two_chars):
            possible_trigrams[trigram] = count
    return possible_trigrams

#### get_possible_trigrams()
The above function takes in the trigram model parameter as well as the last_two_chars parameter, which marks the two characters as found in the generated text, and search through the trigram model, to find all trigrams that begin with the last two chars parameter, giving all possible cmbinations of "next characters" which are then stored in a dictionary

In [25]:
def weighted_random_choice(possible_trigrams):
    third_letters = []
    counts = []
    
    # Extract the third letters and counts from the possible trigrams
    for trigram, count in possible_trigrams.items():
        third_letters.append(trigram[2])
        counts.append(count)
    
    # Randomly select a third letter based on the counts
    if third_letters:  
        return random.choices(third_letters, weights=counts)[0]
    else:
        return None  # No possible trigrams found

#### weighted_random_choice()
The above code takes the dictionary made previosly and extracts the third letter from each trigram, it uses the counts given as weighting for random selection, for example with the two words _"THE"_ and _"THA"_ where _"THE"_ appears 150 times and _"THA"_ appears 75 times, then the probability using the third letter would look like:

    > E::150/X

    > A::75/X


In [26]:
def generate_text(trigram_model, initial_string, length=10000):
    # Initialize with the starting string
    generated_text = initial_string

    while len(generated_text) < length:
        last_two_chars = generated_text[-2:]
        possible_trigrams = get_possible_trigrams(trigram_model, last_two_chars) 
        if not possible_trigrams:
            break # No possible trigrams found

        # Calls the weighted random choice function to select the next character
        next_char = weighted_random_choice(possible_trigrams)

        if next_char:
            # Appends the selected character to the generated text
            generated_text += next_char
        else:
            break # No possible trigrams found

    return generated_text


#### generate_text()
The above code calls all previous functions in a loop to look for the last two characters, find all possible next chars with the _"weighted_random_choice()"_ function, and adds the selected character to the text file, which is specifed as a parameter of 10000, 

# ===================================================================================
# Task 3: Analyze your model

In [27]:
def load_word_list(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        # Create a set of words, stripping whitespace and converting to uppercase
        return set(word.strip().upper() for word in file)

#### load_word_list()
Above is a simple function that takes in the words.txt file, iterates through each line, converting all words to Uppercase to match the trigram model and then returning it as usable data for our comparison

In [28]:
def extract_words(text):
    potential_words = [word.strip('.') for word in text.split() if word.strip('.')]
    # Filters out single characters and retains 'A' and 'I' as they are valid words
    return [word for word in potential_words if len(word) > 1 or word in {'A', 'I'}]

#### extract_words()
This function takes the data we parsed in the generated text, and converts it into valid words, removing whitespace and periods, and also bypassing the rule for 'A' and 'I' characters since they are valid words in the english alphabet

In [29]:
def analyze_words(generated_text, word_list):
    found_words = extract_words(generated_text) # Utilizes the extract_words function to extract the words in the generated text for the comparison 
    real_words = [word for word in found_words if word in word_list]
    
    # Store the number of total words and real words using a length function
    total_words = len(found_words)
    real_word_count = len(real_words)
    if total_words == 0:
        return 0, 0, 0, []

    # Equation to calculate the percentage of real words weighted by the total number of words
    percentage = (real_word_count / total_words) * 100
    return total_words, real_word_count, percentage, real_words


#### analyze_words()
The Above function takes in all the data we've made, and iterates through both files, adding matching words to an array, it calculates the length of all the words and then calculates the percentage of words from the total number of real words over the total words, this data is then returned to be called 

In [30]:
def print_analysis_report(total_words, real_word_count, percentage, real_words, sample_size=40):
    print("Word Analysis Report")
    print("===================")
    print(f"Total potential words found: {total_words}")
    print(f"Real English words found: {real_word_count}")
    print(f"Percentage of real words: {percentage:.2f}%")
    print(f"\nSample of real words found (up to {sample_size}):")
    print(",\n".join(sorted(real_words)[:sample_size]))


#### print_analysis_report()
Basic text output function, outputs a small sample of words

# ===================================================================================
# Task 4: Export your model as JSON

In [31]:
def export_trigrams_to_json(trigram_counts, output_file='trigrams.json'):
    # Convert the trigram counts to a list of dictionaries
    trigram_dict = dict(trigram_counts)

    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(trigram_dict, file, indent=2)

    print(f"Trigram counts exported to {output_file}")

#### export_trigrams_to_json()
The above function takes in the trigrams text and outputs it to JSON, it does this by taking in our trgiram_counts as a regular dict, as JSON cant serialise default one, then we call the json library to serailise the dict to json within the file.

In [ ]:
# Execute Task 1: Process files and create trigram model
base_dir = os.path.join(os.getcwd(), 'englishworks')
if not os.path.exists(base_dir):
    raise FileNotFoundError(f"Directory '{base_dir}' does not exist")

# Process files and create trigram model
for file in os.listdir(base_dir):
    file_path = os.path.join(base_dir, file)
    if os.path.isfile(file_path):
        processed_text = advanced_preprocess_text(file_path)
        with open(combined_processed_file, 'a', encoding='utf-8') as processed_file:
            processed_file.write(f'--- {file} ---\n')
            processed_file.write(processed_text + '\n\n')
        print(f"Processed and saved: {file}")
        global_trigram_counts = trigram_iteration(processed_text, global_trigram_counts)

# Save trigram counts
with open(combined_trigrams_file, 'w', encoding='utf-8') as trigram_file:
    for trigram, count in global_trigram_counts.items():
        trigram_file.write(f'{trigram}: {count}\n')

print(f"All processed texts saved in {combined_processed_file}")
print(f"All trigram counts saved in {combined_trigrams_file}")

# Execute Task 2: Generate text
initial_string = "TH"
generated_text = generate_text(global_trigram_counts, initial_string, length=10000)
print("\nGenerated Text (First 500 characters):\n")
print(generated_text[:500])
with open("generatedTextOutPut.txt", "w") as f:
    f.write(generated_text)

# Execute Task 3: Analyze words
try:
    english_words = load_word_list('words.txt')
    total_words, real_word_count, percentage, real_words = analyze_words(generated_text, english_words)
    print("\nWord Analysis Results:")
    print_analysis_report(total_words, real_word_count, percentage, real_words)
except FileNotFoundError as e:
    print(f"Error loading word list: {str(e)}")

# Execute Task 4: Export to JSON
export_trigrams_to_json(global_trigram_counts, 'trigrams.json')

The Above function is an amalgamantion of method calls to complete the tasks in linear order, , calling each method with respective parameters and outputting debug text for progress reports while the code compiles, the final product is outputted to a file called trgrams.json and can be found in the file hierarchy after compilation